# 🚨 Smart Business Intelligence - Anomaly Detection (Isolation Forest)

**Projet :** Smart Business Intelligence & Sales Analytics Platform  
**Auteur :** Chaima Debchi  
**Objectif :** Détecter automatiquement les transactions statistiquement inhabituelles (pertes financières sévères, remises excessives ou commandes records) grâce à l'algorithme d'apprentissage non supervisé **Isolation Forest**.

---

## 🧠 Comprendre l'Algorithme Isolation Forest
- **Le Principe mathématique :** Contrairement aux méthodes qui modélisent les points normaux, l'Isolation Forest cherche directement à **isoler** les anomalies.
- **Comment ça fonctionne ?** L'algorithme construit un ensemble d'arbres de décision aléatoires (*iTrees*). Comme les anomalies ont des valeurs extrêmes ou rares, elles se retrouvent isolées très rapidement (avec très peu de découpages ou de profondeur dans l'arbre).
- **Score d'anomalie :** Plus le chemin moyen dans l'arbre pour isoler un point est court, plus ce point a un score d'anomalie élevé.

> ⚠️ **Règle Métier Fondamentale :** Une anomalie statistique **n'est pas nécessairement une fraude**.  
> Dans notre dataset commercial, nous distinguons deux grands types d'anomalies :  
> 1. **Les Risques de Rentabilité :** Fortes remises (70-80%) générant des pertes nettes massives (ex: -6 500 $ sur une seule vente).  
> 2. **Les Opportunités Exceptionnelles (*Blockbusters*) :** Très grosses commandes B2B de copieurs ou serveurs générant des profits records (+8 000 $).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.ensemble import IsolationForest
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: "%.2f" % x)

df = pd.read_csv("../data/processed/cleaned_superstore.csv")
print(f"Transactions analysées : {len(df):,}")

## 1. Sélection des Variables & Entraînement du Modèle
Nous sélectionnons les 5 variables clés qui caractérisent le comportement économique d'une transaction :
- `Sales` : Chiffre d'affaires
- `Profit` : Bénéfice net
- `Quantity` : Volume d'articles
- `Discount` : Taux de remise
- `Profit Margin` : Marge nette (%)

In [ ]:
features = ["Sales", "Profit", "Quantity", "Discount", "Profit Margin"]
X = df[features].copy()

# Taux de contamination fixé à 3 % (transactions les plus atypiques)
iso = IsolationForest(contamination=0.03, random_state=42, n_estimators=150)
df["Anomaly_Flag"] = iso.fit_predict(X)
df["Is_Anomaly"] = df["Anomaly_Flag"].map({1: "Normal", -1: "Anomalie"})
df["Anomaly_Score"] = iso.decision_function(X)

print(df["Is_Anomaly"].value_counts())
print(f"Proportion d'anomalies détectées : {(df['Is_Anomaly'] == 'Anomalie').mean()*100:.1f} %")

## 2. Visualisation des Anomalies : Ventes vs Bénéfice

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df,
    x="Sales",
    y="Profit",
    hue="Is_Anomaly",
    palette={"Normal": "#1f77b4", "Anomalie": "#d62728"},
    style="Is_Anomaly",
    alpha=0.8,
    s=60
)
plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.title("Détection des Anomalies : Ventes vs Bénéfice ($)", fontsize=14, fontweight="bold")
plt.xlabel("Chiffre d'Affaires ($)")
plt.ylabel("Bénéfice Net ($)")
plt.tight_layout()
plt.show()

## 3. Typologie des Anomalies : Classification Métier
Pour rendre les alertes actionnables par les équipes opérationnelles, nous classons les 300 anomalies en 3 profils distincts.

In [ ]:
def categorize(row):
    if row["Is_Anomaly"] == "Normal":
        return "Normal"
    if row["Profit"] < -500:
        return "Perte Critique (Marge Négative)"
    elif row["Discount"] >= 0.5:
        return "Remise Excessive (>= 50%)"
    elif row["Sales"] > 3000 and row["Profit"] > 1000:
        return "Vente Exceptionnelle (Blockbuster)"
    else:
        return "Profil Atypique"

df["Anomaly_Type"] = df.apply(categorize, axis=1)
print(df["Anomaly_Type"].value_counts())

### Cas 1 : Les Pertes Financières Critiques (À auditer en urgence)

In [ ]:
losses = df[df["Anomaly_Type"] == "Perte Critique (Marge Négative)"].sort_values("Profit").head(5)
losses[["Order ID", "Customer Name", "State", "Category", "Sub-Category", "Sales", "Profit", "Discount"]]

### Cas 2 : Les Ventes Records (Blockbusters)

In [ ]:
blockbusters = df[df["Anomaly_Type"] == "Vente Exceptionnelle (Blockbuster)"].sort_values("Profit", ascending=False).head(5)
blockbusters[["Order ID", "Customer Name", "State", "Category", "Sub-Category", "Sales", "Profit", "Discount"]]

## 4. Recommandations Pratiques pour le Contrôle de Gestion

1. **Mise en place d'un garde-fou commercial :** Bloquer automatiquement dans l'ERP toute commande avec un taux de remise supérieur à 40% sans validation de la direction financière.
2. **Programme de fidélisation sur les clients Blockbusters :** Suivre personnellement les clients à l'origine des commandes de plus de 10 000 $ (ex: Tamara Chand, Raymond Buch) pour sécuriser leurs renouvellements de contrats.
3. **Audit régional :** Investiguer les pratiques des équipes commerciales de Caroline du Nord et du Texas, où se concentrent les pertes les plus extrêmes.